# Experiment 16: Active-Calibrated & Inactive Tucker Compression (Notebook 02 Method - Kaggle Version)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`, 150 samples)  

### Core Architecture & Formulation:
In Experiment 14/15, compressing inactive weights using aggressive DBSCAN clustering into 10 chunks of 400 with Tucker ranks `[6, 160, 500]` incurred an 84% reconstruction error across all 26 layers, collapsing downstream model performance.

In this experiment:
1. **Notebook 02 Inactive Method as Official Baseline**:
   - The inactive subspace ($4,500$ coordinates/matrix out of 6,912, quarantining 2,412 active coordinates) is processed directly with the proven **Notebook 02 pipeline**:
     - **SVD 95% Spectral Energy Denoising**: Keeps rank $r_{95}$ capturing 95% of singular value energy.
     - **60th Percentile Magnitude Sparsification (Zero-Masking)**: Masks residual noise below the 60th percentile.
     - **4D Balanced Tucker Decomposition**: Reshapes $[4500, 1152]$ to $[45, 100, 24, 48]$ and factorizes with balanced ranks `[30, 45, 16, 32]`.
   - Permanently cuts **349,832,340 parameters** model-wide (86.5% of the inactive slice, 56.3% of all MLP weights)!
   - **This Inactive [Notebook 02 Method] model serves as the official Baseline for all comparisons in this notebook.**
2. **Active Coordinates Strictly Preserved**:
   - The 2,412 most active neurons per submodule are quarantined and preserved in pristine FP32.
3. **Active Subspace Optimization vs Inactive Baseline**:
   - We benchmark against the Inactive Baseline:
     - **Inactive Baseline**: Inactive [Notebook 02 Method] (Active Untouched, 349.8M params cut, $\Delta = +0.00\%$).
     - **Active Variant 1: Permuted High-Capacity Active (`[6, 320, 900]`)**: Active subspace lightly factorized (~392M params cut total).
     - **Active Variant 2: Adam-Calibrated Active Weights**: Active weights of `down_proj` calibrated via 35 Adam GD steps to absorb and cancel residual inactive drift!
4. **Optimized Execution Flow**:
   - Inactive Tucker decompositions are precomputed and cached once across all 78 submodules, eliminating redundant solves and cutting runtime by >50%.

In [ ]:
# Optional: Install required dependencies if not already present in your Kaggle environment
!pip install -q tensorly datasets scikit-learn


In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Standard Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

print("Environment configured.")
print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version: ", torch.__version__)
print("CUDA Available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:     ", torch.cuda.get_device_name(0))


In [ ]:
# =====================================================================
# STEP 2: Model & Dataset Loading (Pure Hugging Face - No Custom Module)
# =====================================================================
import huggingface_hub

MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
NUM_EVAL_SAMPLES = 150

# Hugging Face Authentication for Gated Gemma-3 Model
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Logged into Hugging Face via User Secrets.")
else:
    print("Warning: No HF_TOKEN found. Ensure model is cached or provide token.")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} directly with AutoModelForCausalLM...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=hf_token,
)
model.eval()

# Load 150 GLUE MNLI Validation Samples
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = ds.select(range(NUM_EVAL_SAMPLES))
labels_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [
    tokenizer.encode(" " + name, add_special_tokens=False)[0]
    for name in labels_names
]

print(f"Loaded {MODEL_ID} across {NUM_LAYERS} decoder layers.")
print(f"MNLI Evaluation set: {len(eval_data)} samples. Candidate Token IDs: {label_token_ids}")


In [ ]:
# =====================================================================
# STEP 3: Reference Pristine Model Evaluation & Activation Profiling
# =====================================================================
def evaluate_mnli(model, eval_data, label_token_ids, desc="Evaluating"):
    predictions = []
    ground_truth = []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=desc):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)
            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()
            predictions.append(pred_label)
            ground_truth.append(sample["label"])
    return predictions, ground_truth

print("Evaluating Reference Pristine Uncompressed Model...")
pristine_preds, pristine_gts = evaluate_mnli(model, eval_data, label_token_ids, desc="Evaluating Pristine Reference")
pristine_accuracy = accuracy_score(pristine_gts, pristine_preds)
print(f"Reference Pristine Accuracy (150 samples): {pristine_accuracy * 100:.2f}%")

# Cache pristine weights on CPU
W_orig_all = {}
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": lmod.gate_proj.weight.data.clone().cpu(),
        "up_proj": lmod.up_proj.weight.data.clone().cpu(),
        "down_proj": lmod.down_proj.weight.data.clone().cpu(),
    }

# Forward hooks for MLP activations
layer_acts = {l: {"gate_proj": [], "up_proj": [], "down_proj": [], "mlp_input": []} for l in range(NUM_LAYERS)}
hooks = []

for l in range(NUM_LAYERS):
    mlp = model.model.layers[l].mlp
    def make_hook(layer_idx, name):
        def hook(m, inp, out):
            if name == "down_proj":
                layer_acts[layer_idx][name].append(inp[0].detach().cpu().squeeze(0).mean(dim=0))
            elif name == "mlp_input":
                layer_acts[layer_idx][name].append(inp[0].detach().cpu().squeeze(0).mean(dim=0))
            else:
                act = out[0] if isinstance(out, tuple) else out
                layer_acts[layer_idx][name].append(act.detach().cpu().squeeze(0).mean(dim=0))
        return hook

    hooks.append(mlp.act_fn.register_forward_hook(make_hook(l, "gate_proj")))
    hooks.append(mlp.up_proj.register_forward_hook(make_hook(l, "up_proj")))
    hooks.append(mlp.down_proj.register_forward_hook(make_hook(l, "down_proj")))
    hooks.append(mlp.register_forward_hook(make_hook(l, "mlp_input")))

print("Profiling MLP activations across 20 calibration samples...")
with torch.no_grad():
    for i in range(20):
        sample = eval_data[i]
        prompt = f"<start_of_turn>user\nPremise: {sample['premise']}\nHypothesis: {sample['hypothesis']}\n<start_of_turn>model\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        model(**inputs, logits_to_keep=1)

for h in hooks:
    h.remove()

acts_matrix_all = {l: {} for l in range(NUM_LAYERS)}
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        acts_matrix_all[l][sub_name] = torch.stack(layer_acts[l][sub_name], dim=0).numpy()

print(f"Successfully captured activation statistics for all 78 submodules ({acts_matrix_all[0]['gate_proj'].shape}).")


In [ ]:
# =====================================================================
# STEP 4: Inactive Coordinate Partition (Notebook 02 Method) & Active Setup
# =====================================================================
NUM_INACTIVE = 4500
NUM_ACTIVE = 2412   # 6,912 - 4,500 = 2,412 active coords kept pristine
CHUNK_SIZE = 400
ACT_NUM_CHUNKS = 6  # 2,400 active coords factorized in Active Variant 1

layer_data = {}

def process_submodule_partition(acts_matrix, weight_tensor, is_col=False, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    v_all = np.var(acts_matrix, axis=0)
    sorted_all = np.argsort(v_all)
    inactive_pool = sorted_all[:NUM_INACTIVE] # 4,500 least active coords
    active_pool = sorted_all[NUM_INACTIVE:]   # 2,412 active coords
    
    # Active Subspace Setup (Permuted 6 chunks of size 400 for Variant 1)
    v_act = np.mean(np.abs(acts_matrix[:, active_pool]), axis=0)
    eps_act = max(0.04, float(np.std(v_act) * 0.18))
    db_act = DBSCAN(eps=eps_act, min_samples=30, metric="euclidean")
    act_labels = db_act.fit_predict(v_act.reshape(-1, 1))
    
    act_chunks = []
    for lab in [l for l in np.unique(act_labels) if l != -1]:
        c_idx = np.where(act_labels == lab)[0]
        if len(c_idx) == 0: continue
        c_coords = active_pool[c_idx]
        sorted_c = c_coords[np.argsort(v_all[c_coords])]
        for ci in range(len(sorted_c) // CHUNK_SIZE):
            act_chunks.append(sorted_c[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(act_chunks) >= ACT_NUM_CHUNKS: break
        if len(act_chunks) >= ACT_NUM_CHUNKS: break
        
    if len(act_chunks) < ACT_NUM_CHUNKS:
        assigned = set(np.concatenate(act_chunks) if act_chunks else [])
        avail = [c for c in active_pool if c not in assigned]
        needed_act = ACT_NUM_CHUNKS - len(act_chunks)
        for ci in range(needed_act):
            if len(avail) >= CHUNK_SIZE:
                act_chunks.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]
                
    if is_col:
        T_act = torch.stack([weight_tensor[:, c].T.float().cpu() for c in act_chunks], dim=0)
    else:
        T_act = torch.stack([weight_tensor[c, :].float().cpu() for c in act_chunks], dim=0)
        
    return {
        "inactive_pool": inactive_pool,
        "active_pool": active_pool,
        "act_chunks": act_chunks,
        "T_act": T_act,
        "is_col": is_col,
    }

print("Running coordinate partition across all 26 layers...")
for l in tqdm(range(NUM_LAYERS), desc="Partitioning Submodules"):
    layer_data[l] = {}
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        layer_data[l][sub_name] = process_submodule_partition(
            acts_matrix_all[l][sub_name],
            W_orig_all[l][sub_name],
            is_col=(sub_name == "down_proj"),
            device=sub_mod.weight.device,
        )

print("Coordinate partition complete: 4,500 inactive coords / 2,412 active coords per submodule.")


In [ ]:
# =====================================================================
# STEP 5: Define GPU Tucker GD Optimizer & Active Adam Calibrator
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    safe_ranks = [
        min(ranks[0], T.shape[0] - 1),
        min(ranks[1], T.shape[1] - 1),
        min(ranks[2], T.shape[2] - 1),
    ]
    T_target = T.to(device)
    core_init, factors_init = tucker(T_target, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone())
    factors_param = [torch.nn.Parameter(f.clone()) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param))
        final_err = (torch.norm(T_target - T_recon_final) / torch.norm(T_target)).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

def calibrate_active_weights_mlp(lmod, sdata_layer, orig_w_layer, layer_mlp_inputs, num_steps=35, lr=1e-3, device=None):
    """
    Calibrates the active weights of down_proj to minimize the residual output error
    introduced by compressing the inactive weights of gate_proj, up_proj, and down_proj.
    Runs fast local Adam GD directly on GPU (< 0.05 seconds per layer).
    """
    if device is None:
        device = lmod.down_proj.weight.device
        
    sdata_down = sdata_layer["down_proj"]
    act_indices = sdata_down["active_pool"]
    
    # Representative MLP input activations: Shape [150, 1152]
    x_sample = torch.stack(layer_mlp_inputs, dim=0).to(device, dtype=torch.float32)
    
    with torch.no_grad():
        # Pristine uncompressed MLP target output
        gate_orig = lmod.act_fn(torch.matmul(x_sample, orig_w_layer["gate_proj"].to(device, dtype=torch.float32).t()))
        up_orig = torch.matmul(x_sample, orig_w_layer["up_proj"].to(device, dtype=torch.float32).t())
        inter_orig = gate_orig * up_orig
        target_output = torch.matmul(inter_orig, orig_w_layer["down_proj"].to(device, dtype=torch.float32).t())
        
        # Compressed intermediate activations through current gate_proj and up_proj
        gate_comp = lmod.act_fn(torch.matmul(x_sample, lmod.gate_proj.weight.to(device, dtype=torch.float32).t()))
        up_comp = torch.matmul(x_sample, lmod.up_proj.weight.to(device, dtype=torch.float32).t())
        inter_comp = (gate_comp * up_comp).detach()
        
        # Uncalibrated baseline output of down_proj
        base_output = torch.matmul(inter_comp, lmod.down_proj.weight.to(device, dtype=torch.float32).t())
        
        # Residual error to cancel via active weights
        residual_target = (target_output - base_output).detach()
        inter_act = inter_comp[:, act_indices].detach()

    # Parameter: Delta for active columns of down_proj [1152, len(act_indices)]
    delta_active = nn.Parameter(torch.zeros(1152, len(act_indices), device=device, dtype=torch.float32))
    optimizer = torch.optim.Adam([delta_active], lr=lr, weight_decay=1e-4)

    for step in range(num_steps):
        optimizer.zero_grad()
        pred_delta = torch.matmul(inter_act, delta_active.t())
        loss = torch.norm(residual_target - pred_delta) ** 2
        loss.backward()
        optimizer.step()

    # In-place update active columns with calibrated delta
    with torch.no_grad():
        lmod.down_proj.weight.data[:, act_indices] += delta_active.data.to(dtype=lmod.down_proj.weight.dtype)

print("Defined GPU Tucker GD optimizer and Active Weight Adam Calibrator.")


In [ ]:
# =====================================================================
# STEP 6: Precompute Inactive [Notebook 02 Method] & Inactive Baseline Evaluation
# =====================================================================
RANKS_02 = [30, 45, 16, 32] # Balanced ranks from Notebook 02

# --- 6.1: Precompute and Cache Inactive [Notebook 02 Method] Factorizations ---
print("Precomputing Inactive Notebook 02 factorizations across all 78 submodules...")
inact_recon_cache = {l: {} for l in range(NUM_LAYERS)}
inact_gate_errs, inact_up_errs, inact_down_errs = [], [], []
total_inact_params_saved = 0

for l in tqdm(range(NUM_LAYERS), desc="Notebook 02 Inactive Precomputation"):
    sdata_layer = layer_data[l]
    lmod = model.model.layers[l].mlp
    inactive_idx = sdata_layer["gate_proj"]["inactive_pool"]
    
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = sdata_layer[sub_name]
        is_col = sdata["is_col"]
        orig_w = W_orig_all[l][sub_name]
        sub_dev = getattr(lmod, sub_name).weight.device
        
        W_inact = (orig_w[:, inactive_idx].T if is_col else orig_w[inactive_idx, :]).float().to(sub_dev)
        
        # 1. SVD 95% Energy Denoising
        U, S, Vh = torch.linalg.svd(W_inact, full_matrices=False)
        cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
        r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
        W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
        
        # 2. 60th Percentile Magnitude Sparsification (Zero-Masking)
        eps = torch.quantile(torch.abs(W_denoised), 0.60)
        W_sparse = W_denoised.clone()
        W_sparse[torch.abs(W_sparse) < eps] = 0.0
        
        # 3. 4D Tucker Decomposition: [4500, 1152] -> [45, 100, 24, 48]
        W_tensor = W_sparse.reshape(45, 100, 24, 48)
        core, factors = tucker(W_tensor, rank=RANKS_02, init='svd')
        W_recon = tucker_to_tensor((core, factors)).reshape(NUM_INACTIVE, 1152)
        
        err = (torch.norm(W_inact - W_recon) / torch.norm(W_inact)).item()
        inact_recon_cache[l][sub_name] = W_recon.cpu()
        
        if sub_name == "gate_proj": inact_gate_errs.append(err)
        elif sub_name == "up_proj":  inact_up_errs.append(err)
        elif sub_name == "down_proj": inact_down_errs.append(err)
        
        orig_p = NUM_INACTIVE * 1152 # 5,184,000
        comp_p = core.numel() + sum(f.numel() for f in factors) # 698,970
        total_inact_params_saved += (orig_p - comp_p)

mean_gate = np.mean(inact_gate_errs) * 100
mean_up   = np.mean(inact_up_errs) * 100
mean_down = np.mean(inact_down_errs) * 100
print(f"Inactive Notebook 02 precomputation complete! Total inactive params cut: {total_inact_params_saved:,}")
print(f"  Mean Inactive Recon Errors: gate={mean_gate:.1f}%, up={mean_up:.1f}%, down={mean_down:.1f}%")

# --- 6.2: Inject Inactive Compressed Weights & Evaluate Inactive Baseline ---
print("\nInjecting Inactive Notebook 02 weights across all 78 submodules (Active untouched)...")
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    sdata_layer = layer_data[l]
    inactive_idx = sdata_layer["gate_proj"]["inactive_pool"]
    active_idx = sdata_layer["gate_proj"]["active_pool"]
    
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = sdata_layer[sub_name]
        mod_ref = getattr(lmod, sub_name)
        sub_dev = mod_ref.weight.device
        orig_w = W_orig_all[l][sub_name]
        W_recon = inact_recon_cache[l][sub_name].to(sub_dev)
        
        mod_ref.weight.data = orig_w.clone().to(sub_dev)
        if sdata["is_col"]:
            mod_ref.weight.data[:, inactive_idx] = W_recon.T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[:, active_idx] = orig_w[:, active_idx].to(sub_dev)
        else:
            mod_ref.weight.data[inactive_idx, :] = W_recon.to(device=sub_dev, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[active_idx, :] = orig_w[active_idx, :].to(sub_dev)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Evaluate Inactive Baseline Accuracy
print("Evaluating Inactive Baseline [Notebook 02 Method] (Active Untouched)...")
inact_preds, inact_gts = evaluate_mnli(model, eval_data, label_token_ids, desc="Evaluating Inactive Baseline")
inactive_baseline_accuracy = accuracy_score(inact_gts, inact_preds)
baseline_accuracy = inactive_baseline_accuracy

print(f"\n===============================================================================================")
print(f"OFFICIAL EXPERIMENT BASELINE: Inactive Baseline [Notebook 02 Method] (Active Untouched)")
print(f"  Downstream Accuracy: {inactive_baseline_accuracy * 100:.2f}% (Pristine Reference: {pristine_accuracy * 100:.2f}%)")
print(f"  Total Params Cut:    {total_inact_params_saved:,}")
print(f"===============================================================================================")


In [ ]:
# =====================================================================
# STEP 7: Active Variant 1 - Permuted Active Tucker ([6, 320, 900])
# =====================================================================
ACTIVE_RANKS_V1 = [6, 320, 900]
print(f"Testing Active Variant 1: Permuted Active Tucker with ranks {ACTIVE_RANKS_V1}...")

act_params_saved_v1 = 0
active_v1_gate_errs, active_v1_up_errs, active_v1_down_errs = [], [], []

for l in tqdm(range(NUM_LAYERS), desc="Active Variant 1 Optimization"):
    lmod = model.model.layers[l].mlp
    sdata_layer = layer_data[l]
    inactive_idx = sdata_layer["gate_proj"]["inactive_pool"]
    
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = sdata_layer[sub_name]
        mod_ref = getattr(lmod, sub_name)
        sub_dev = mod_ref.weight.device
        orig_w = W_orig_all[l][sub_name]
        
        # Reset and inject cached inactive weights
        mod_ref.weight.data = orig_w.clone().to(sub_dev)
        W_inact_recon = inact_recon_cache[l][sub_name].to(sub_dev)
        if sdata["is_col"]:
            mod_ref.weight.data[:, inactive_idx] = W_inact_recon.T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
        else:
            mod_ref.weight.data[inactive_idx, :] = W_inact_recon.to(device=sub_dev, dtype=mod_ref.weight.dtype)
            
        # Optimize Active Tucker
        T_act = sdata["T_act"]
        cg, fg, T_act_recon, err = optimize_tucker_gd(
            T_act, ranks=ACTIVE_RANKS_V1, num_steps=35, lr=1e-3, device=sub_dev
        )
        if sub_name == "gate_proj": active_v1_gate_errs.append(err)
        elif sub_name == "up_proj":  active_v1_up_errs.append(err)
        elif sub_name == "down_proj": active_v1_down_errs.append(err)
        
        orig_act_p = T_act.numel()
        comp_act_p = cg.numel() + sum(f.numel() for f in fg)
        act_params_saved_v1 += (orig_act_p - comp_act_p)
        
        # Inject Active Factorization
        if sdata["is_col"]:
            for k, c in enumerate(sdata["act_chunks"]):
                mod_ref.weight.data[:, c] = T_act_recon[k].T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
        else:
            for k, c in enumerate(sdata["act_chunks"]):
                mod_ref.weight.data[c, :] = T_act_recon[k].to(device=sub_dev, dtype=mod_ref.weight.dtype)

total_params_cut_v1 = total_inact_params_saved + act_params_saved_v1
v1_preds, v1_gts = evaluate_mnli(model, eval_data, label_token_ids, desc="Evaluating Active Variant 1")
v1_accuracy = accuracy_score(v1_gts, v1_preds)

print(f"Active Variant 1 Accuracy: {v1_accuracy * 100:.2f}% (Delta vs Baseline: {(v1_accuracy - baseline_accuracy) * 100:+.2f}%)")
print(f"Total Params Cut: {total_params_cut_v1:,}")


In [ ]:
# =====================================================================
# STEP 8: Active Variant 2 - Adam-Calibrated Active Weights
# =====================================================================
print("Testing Active Variant 2: Adam-Calibrated Active Weights across all 26 layers...")

for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    sdata_layer = layer_data[l]
    inactive_idx = sdata_layer["gate_proj"]["inactive_pool"]
    active_idx = sdata_layer["gate_proj"]["active_pool"]
    
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = sdata_layer[sub_name]
        mod_ref = getattr(lmod, sub_name)
        sub_dev = mod_ref.weight.device
        orig_w = W_orig_all[l][sub_name]
        
        # Re-inject Inactive Baseline weights
        mod_ref.weight.data = orig_w.clone().to(sub_dev)
        W_inact_recon = inact_recon_cache[l][sub_name].to(sub_dev)
        if sdata["is_col"]:
            mod_ref.weight.data[:, inactive_idx] = W_inact_recon.T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[:, active_idx] = orig_w[:, active_idx].to(sub_dev)
        else:
            mod_ref.weight.data[inactive_idx, :] = W_inact_recon.to(device=sub_dev, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[active_idx, :] = orig_w[active_idx, :].to(sub_dev)

# Run Local GPU Adam GD Calibration on Active Columns of down_proj
for l in tqdm(range(NUM_LAYERS), desc="Active Adam Calibration"):
    lmod = model.model.layers[l].mlp
    calibrate_active_weights_mlp(
        lmod,
        layer_data[l],
        W_orig_all[l],
        layer_acts[l]["mlp_input"],
        num_steps=35,
        lr=1e-3,
        device=lmod.down_proj.weight.device,
    )

if torch.cuda.is_available():
    torch.cuda.empty_cache()

v2_preds, v2_gts = evaluate_mnli(model, eval_data, label_token_ids, desc="Evaluating Active Variant 2")
v2_accuracy = accuracy_score(v2_gts, v2_preds)

print(f"Active Variant 2 Accuracy: {v2_accuracy * 100:.2f}% (Delta vs Baseline: {(v2_accuracy - baseline_accuracy) * 100:+.2f}%)")
print(f"Total Params Cut: {total_inact_params_saved:,}")


In [ ]:
# =====================================================================
# STEP 9: Final Comparison Table, Visualizations & Results Export
# =====================================================================
results_table = [
    {
        "variant": "Reference: Pristine Uncompressed Model",
        "gate_err": 0.0,
        "up_err": 0.0,
        "down_err": 0.0,
        "params_cut": 0,
        "accuracy": pristine_accuracy,
        "delta": pristine_accuracy - baseline_accuracy,
    },
    {
        "variant": "Inactive Baseline [Notebook 02 Method] (Active Untouched)",
        "gate_err": mean_gate,
        "up_err": mean_up,
        "down_err": mean_down,
        "params_cut": total_inact_params_saved,
        "accuracy": baseline_accuracy,
        "delta": 0.0,
    },
    {
        "variant": "Active Variant 1: Permuted Active Tucker ([6, 320, 900])",
        "gate_err": mean_gate,
        "up_err": mean_up,
        "down_err": mean_down,
        "params_cut": total_params_cut_v1,
        "accuracy": v1_accuracy,
        "delta": v1_accuracy - baseline_accuracy,
    },
    {
        "variant": "Active Variant 2: Adam-Calibrated Active Weights",
        "gate_err": mean_gate,
        "up_err": mean_up,
        "down_err": mean_down,
        "params_cut": total_inact_params_saved,
        "accuracy": v2_accuracy,
        "delta": v2_accuracy - baseline_accuracy,
    },
]

print("=" * 125)
print(f"{'Variant':<65} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<12} | {'Accuracy':<9} | {'Delta':<7}")
print("=" * 125)
for r in results_table:
    delta_str = f"{r['delta'] * 100:+.2f}%" if r['variant'] != results_table[0]['variant'] else "---"
    print(
        f"{r['variant']:<65} | "
        f"{r['gate_err']:>6.2f}%  | "
        f"{r['up_err']:>5.2f}%  | "
        f"{r['down_err']:>6.2f}%   | "
        f"{r['params_cut']:<12,d} | "
        f"{r['accuracy'] * 100:>7.2f}% | "
        f"{delta_str:>7}"
    )
print("=" * 125)

# Plot Results
variants = [r["variant"].replace("Active ", "").replace(": ", "\n") for r in results_table]
accs = [r["accuracy"] * 100 for r in results_table]
cuts = [r["params_cut"] / 1e6 for r in results_table]

fig, ax1 = plt.subplots(figsize=(11, 5))
color = 'tab:blue'
ax1.set_xlabel('Compression Variant', fontweight='bold')
ax1.set_ylabel('Accuracy (%)', color=color, fontweight='bold')
bars = ax1.bar(variants, accs, color=color, alpha=0.6, width=0.45)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 60)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.0, f"{acc:.2f}%", ha='center', fontweight='bold')

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Parameters Cut (Millions)', color=color, fontweight='bold')
ax2.plot(variants, cuts, color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, 450)
for i, cut in enumerate(cuts):
    ax2.text(i, cut + 15, f"{cut:.1f}M", ha='center', color=color, fontweight='bold')

plt.title("Experiment 16: Accuracy vs Parameter Reduction (Notebook 02 Inactive Method)", fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Export Results to JSON
artifacts_dir = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("./artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "16_active_calibrated_inactive_results.json"
with open(results_file, "w") as f:
    json.dump({
        "experiment": "16_active_calibrated_inactive_tucker_02_method",
        "target_model": MODEL_ID,
        "inactive_method": "notebook_02_4d_tucker_[30,45,16,32]",
        "results": results_table,
    }, f, indent=2)

print(f"\nSaved active-calibrated benchmark results to {results_file}")
